# 02 — Endemic-species verification

**Purpose:** connect cleaned bird records to sourced species traits and identify the endemic-only candidate dataset for habitat-risk assessment.

Run each cell yourself. All processing code is shown here. This notebook does not download traits, invent missing status, calculate weights or score HRI. The existing trait evidence is partially verified and reflects recorded source snapshots, not automatically current or historical assessment status.

Notebook 01 flags remain attached. We use its all-row cleaning draft; we do not silently adopt the deduplicated candidate or remove every flagged record. “Endemic candidate” means scope eligibility, not final record quality or completed trait verification.

## 1. Load the cleaning draft and trait evidence

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data/metadata/species_traits.csv").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside component4.")
    ROOT = ROOT.parent

birds = pd.read_csv(ROOT / "data/interim/hri/bird_observations_cleaning_draft.csv")
traits = pd.read_csv(ROOT / "data/metadata/species_traits.csv")
print("Bird records:", len(birds))
print("Species evidence rows:", len(traits))
traits[["scientific_name", "endemic_status", "iucn_category", "verification_status"]]

## 2. Check the species join key
Use the recorded scientific name for matching. Accepted names are retained separately; replacing names before verifying the crosswalk can lose matches or merge different taxonomic concepts.

In [ ]:
birds["scientific_name"] = birds["scientific_name"].astype("string").str.strip()
traits["scientific_name"] = traits["scientific_name"].astype("string").str.strip()

assert traits["scientific_name"].notna().all(), "Trait names are missing."
assert traits["scientific_name"].is_unique, "Resolve duplicate trait keys before joining."

unmatched_names = birds.loc[
    ~birds["scientific_name"].isin(traits["scientific_name"]), "scientific_name"
].drop_duplicates()
print("Unmatched names:", unmatched_names.tolist())

## 3. Join observations to species evidence
Many observations may belong to one species. `validate="many_to_one"` prevents a duplicate trait key from multiplying observation rows. Keep common names from observations and retain trait provenance.

In [ ]:
trait_columns = [
    "scientific_name", "accepted_scientific_name", "endemic_status", "endemic_source",
    "iucn_category", "iucn_source", "iucn_snapshot_year", "iucn_assessment_year",
    "forest_dependence", "forest_dependence_source", "taxonomy_match_status",
    "verification_status", "inclusion_decision", "notes"
]
joined = birds.merge(
    traits[trait_columns], on="scientific_name", how="left",
    validate="many_to_one", indicator="trait_match"
)
assert len(joined) == len(birds), "The join changed the number of observations."
print(joined["trait_match"].value_counts())

## 4. Make the endemic-only scope decision visible
The study is about Sri Lankan endemic birds. Sourced `yes` records become candidates, `no` records stay in a separate outside-scope table, and unknown status remains pending. These new labels do not overwrite the master trait table or its pending review decisions.

In [ ]:
status = joined["endemic_status"].astype("string").str.strip().str.lower()
joined["scope_decision"] = "pending_endemism_evidence"
joined.loc[status.eq("yes").fillna(False), "scope_decision"] = "endemic_candidate"
joined.loc[status.eq("no").fillna(False), "scope_decision"] = "outside_endemic_scope"

scope_summary = joined.groupby("scope_decision").agg(
    observation_rows=("scientific_name", "size"),
    species_names=("scientific_name", "nunique")
)
scope_summary

## 5. Keep missing or conflicting traits visible
A missing IUCN category is not Least Concern. A category is not a numeric vulnerability weight. The stored taxonomy mismatch requires review; do not borrow the category of a broader parent taxon. Recorded source dates also cannot automatically be used as historical predictors.

In [ ]:
joined["iucn_missing"] = joined["iucn_category"].isna()
joined["forest_dependence_missing"] = joined["forest_dependence"].isna()
joined["taxonomy_needs_review"] = (
    joined["taxonomy_match_status"].eq("taxon_concept_mismatch")
    | joined["taxonomy_match_status"].isna()
)
joined["trait_review_needed"] = (
    joined["iucn_missing"] | joined["forest_dependence_missing"]
    | joined["taxonomy_needs_review"]
    | joined["verification_status"].ne("verified")
)

species_review = joined.loc[joined["trait_review_needed"], trait_columns].drop_duplicates()
species_review[["scientific_name", "endemic_status", "iucn_category",
                "forest_dependence", "taxonomy_match_status", "verification_status"]]

## 6. Separate the drafts without losing records
All earlier record-quality flags remain in these tables. Endemic candidates with incomplete traits remain visible rather than being assigned fabricated risk scores. Some traits may be unnecessary for a future HRI definition; decide that explicitly later.

In [ ]:
endemic = joined.loc[joined["scope_decision"].eq("endemic_candidate")].copy()
outside_scope = joined.loc[joined["scope_decision"].eq("outside_endemic_scope")].copy()
pending = joined.loc[joined["scope_decision"].eq("pending_endemism_evidence")].copy()

assert len(endemic) + len(outside_scope) + len(pending) == len(birds)
print("Endemic candidate records:", len(endemic))
print("Endemic candidate species:", endemic["scientific_name"].nunique())
print("Record-quality review flags retained:", endemic["needs_review"].value_counts().to_dict())
print("Outside-scope species:", outside_scope["scientific_name"].drop_duplicates().tolist())

## 7. Save only after reviewing the outputs
These are interim HRI preparation files, not final HRI labels or training data. The raw data and master trait evidence remain unchanged.

In [ ]:
OUT = ROOT / "data/interim/hri"
OUT.mkdir(parents=True, exist_ok=True)
endemic.to_csv(OUT / "endemic_bird_candidates.csv", index=False)
outside_scope.to_csv(OUT / "birds_outside_endemic_scope.csv", index=False)
pending.to_csv(OUT / "birds_pending_endemism_review.csv", index=False)
species_review.to_csv(OUT / "species_traits_for_review.csv", index=False)
scope_summary.to_csv(OUT / "endemic_scope_summary.csv")
print("Saved reviewable drafts to:", OUT)

## Explain this to your lecturer

“I joined the observations to a sourced species table without duplicating records. I separated non-endemic species because they are outside the study scope, preserved unknown cases and record-quality flags, and did not convert incomplete traits into vulnerability scores.”

**Next:** resolve cleaning decisions and map candidate observations to grid cells while preserving observation dates and effort. We then define supported environmental risk indicators and a justified HRI. We do not use the archived forest-loss label as HRI.

Read `data/metadata/species_traits_README.md` for sources and limitations. No fresh verification of current conservation status is claimed by this notebook.